In [1]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

In [2]:
from tempfile import TemporaryDirectory
from PIL import Image
from klygo import datasets, files, media

if 'tutorial_workspace' in globals():
    tutorial_workspace.cleanup()
tutorial_workspace = TemporaryDirectory(prefix='klygo-datasets-convert-')
root = files.path(tutorial_workspace.name)
print('Initialized temporary tutorial workspace:', root)

Initialized temporary tutorial workspace: C:\Users\Mine\AppData\Local\Temp\klygo-datasets-convert-9b412_wz


# Convert annotation formats to YOLO

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/datasets/Document_Datasets_Convert.ipynb)

## Jump to an API

- [datasets.convert](#datasets.convert)

### datasets.convert

`datasets.convert(source=None, target="", *, src_format="voc", dst_format="yolo", images=None, annotations=None, classes=None, image_ids=None, include_difficult=True, include_crowd=False, voc_1_based=False, overwrite=False, verbose=False) -> pathlib.Path`

- Converts Pascal VOC XML or COCO instances JSON to flat YOLO format.
- Preserves original image bytes without re-encoding.
- Maps sparse COCO categories to contiguous 0-indexed YOLO classes.
- Retains negative background images with empty label files.

> **Note**: For directory tree layout flattening, use `datasets.flatten()`.

#### Basic use: Pascal VOC XML conversion

In [3]:
voc_source = files.join(root, 'voc_raw')
images_dir = files.join(voc_source, 'JPEGImages')
ann_dir = files.join(voc_source, 'Annotations')
files.mkdir(images_dir)
files.mkdir(ann_dir)

media.save(files.join(images_dir, '000001.jpg'), Image.new('RGB', (200, 100), 'white'), verbose=False)
xml_content = """<annotation>
    <size><width>200</width><height>100</height></size>
    <object>
        <name>car</name>
        <difficult>0</difficult>
        <bndbox><xmin>20</xmin><ymin>10</ymin><xmax>100</xmax><ymax>60</ymax></bndbox>
    </object>
</annotation>"""
files.save(files.join(ann_dir, '000001.xml'), xml_content, verbose=False)

yolo_target = files.join(root, 'yolo_flat')
datasets.convert(source=voc_source, target=yolo_target, src_format='voc', classes=['car'])
print(datasets.info(yolo_target))

{'classes': ['car'], 'class_ids': [0], 'nc': 1, 'total_images': 1, 'total_labels': 1, 'unlabeled_images': 0, 'splits': {'raw': 1}}


#### COCO instances JSON conversion

In [4]:
coco_source = files.join(root, 'coco_raw')
coco_images = files.join(coco_source, 'images')
files.mkdir(coco_images)
media.save(files.join(coco_images, 'frame1.jpg'), Image.new('RGB', (300, 200), 'gray'), verbose=False)

coco_json = {
    'images': [{'id': 1, 'file_name': 'frame1.jpg', 'width': 300, 'height': 200}],
    'categories': [{'id': 7, 'name': 'car'}],
    'annotations': [{'id': 1, 'image_id': 1, 'category_id': 7, 'bbox': [30, 20, 100, 80], 'iscrowd': 0}],
}
files.save(files.join(coco_source, 'instances.json'), coco_json, verbose=False)

coco_yolo = files.join(root, 'coco_yolo')
datasets.convert(source=coco_source, target=coco_yolo, src_format='coco')
print('COCO converted classes:', datasets.info(coco_yolo)['classes'])

COCO converted classes: ['car']


#### Expected error on deprecated layout parameter

Passing `tree` to `datasets.convert()` raises a `TypeError` directing you to `datasets.flatten()`.

In [5]:
try:
    datasets.convert(voc_source, yolo_target, tree={'images': 'JPEGImages'})
except TypeError as exc:
    print('Caught expected TypeError:', exc)

Caught expected TypeError: datasets.convert() is now dedicated to annotation format conversion (VOC/COCO -> YOLO). For dataset layout flattening with directory trees, use datasets.flatten().


## Put it together

Convert external annotation formats into standard YOLO format, then load or partition seamlessly.

In [6]:
loaded = datasets.load(yolo_target, format='yolo', verbose=False)
print('Total images ready for evaluation or training:', len(loaded))

Total images ready for evaluation or training: 1


## Remember

Annotation conversion preserves exact image bytes, handles negative images, and maps classes contiguously. For layout restructuring, see [flatten](Document_Datasets_Flatten.ipynb) or return to [the datasets overview](Document_Datasets_Overview.ipynb).